# Epoch Analysis Notebook

Interactive exploration of how training epochs affect model performance.

Run `python src/main.py` first to generate `outputs/history.json` and checkpoints.

In [ ]:
import sys, os
# Allow imports from src/
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import json
import glob
import numpy as np
import matplotlib.pyplot as plt
import torch

from data      import get_dataloaders
from model     import build_model
from utils     import get_device, load_history, seed_everything
from visualize import plot_loss_curve, plot_accuracy_curve, plot_decision_boundary, plot_epoch_snapshots

seed_everything(42)
device = get_device()
print('Device:', device)

## 1. Load Training History

In [ ]:
history = load_history('outputs')
print('Epochs recorded:', len(history['train_loss']))
print('Final train loss:', f"{history['train_loss'][-1]:.4f}")
print('Final val loss:  ', f"{history['val_loss'][-1]:.4f}")
print('Final val acc:   ', f"{history['val_acc'][-1]:.4f}")

## 2. Loss and Accuracy Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs, history['train_loss'], label='Train',      color='#2196F3', lw=2)
axes[0].plot(epochs, history['val_loss'],   label='Validation', color='#F44336', lw=2, ls='--')
axes[0].set_title('Loss vs. Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, [a*100 for a in history['train_acc']], label='Train',      color='#4CAF50', lw=2)
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   label='Validation', color='#FF9800', lw=2, ls='--')
axes[1].set_title('Accuracy vs. Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 105)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Identify the Overfitting Point

The overfitting point is where validation loss starts to diverge from training loss.

In [ ]:
val_losses = history['val_loss']
best_epoch = int(np.argmin(val_losses)) + 1
best_val_loss = min(val_losses)
best_val_acc  = history['val_acc'][best_epoch - 1]

print(f'Best epoch (lowest val loss): {best_epoch}')
print(f'Best val loss:  {best_val_loss:.4f}')
print(f'Best val acc:   {best_val_acc:.4f}')

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(val_losses)+1), val_losses, color='#F44336', lw=2, label='Val loss')
plt.axvline(best_epoch, color='green', lw=1.5, ls='--', label=f'Best epoch = {best_epoch}')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Finding the Optimal Stopping Point')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Decision Boundary at Key Epochs

Load checkpoints and compare boundaries side by side.

In [ ]:
_, _, X_all, y_all = get_dataloaders(n_samples=1000, noise=0.20, random_state=42)
model = build_model('medium').to(device)

checkpoint_files = sorted(glob.glob('checkpoints/epoch_*.pt'))
if checkpoint_files:
    out = plot_epoch_snapshots(model, checkpoint_files, X_all, y_all, device=device,
                               output_dir='outputs', show=True)
    print('Saved:', out)
else:
    print('No checkpoints found. Run python src/main.py first.')

## 5. Overfitting Experiment

Train a large model for many epochs on clean data to induce overfitting.

In [ ]:
from train import train as train_model

train_loader, val_loader, X_all, y_all = get_dataloaders(
    n_samples=500, noise=0.05, random_state=42
)

overfit_model = build_model('overfit').to(device)
overfit_history = train_model(
    model=overfit_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=300,
    lr=1e-2,
    device=device,
    checkpoint_dir='checkpoints_overfit',
    verbose=True,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, 301)
axes[0].plot(epochs, overfit_history['train_loss'], label='Train',      color='#2196F3', lw=2)
axes[0].plot(epochs, overfit_history['val_loss'],   label='Validation', color='#F44336', lw=2, ls='--')
axes[0].set_title('Overfitting Model - Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [a*100 for a in overfit_history['train_acc']], label='Train',      color='#4CAF50', lw=2)
axes[1].plot(epochs, [a*100 for a in overfit_history['val_acc']],   label='Validation', color='#FF9800', lw=2, ls='--')
axes[1].set_title('Overfitting Model - Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Note: Watch for the validation accuracy dropping while train accuracy keeps rising.')

## Summary

| Observation | Diagnosis | Remedy |
|---|---|---|
| Both losses high and flat | Underfitting | More epochs, bigger model |
| Val loss rising, train falling | Overfitting | Fewer epochs, regularisation |
| Both losses low and stable | Well-fitted | This is the goal |
| Large train/val accuracy gap | Overfitting | Dropout, weight decay, early stopping |